### Nhập dữ liệu

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("forestfires.csv")

y = np.log(1 + df["area"].values)

X_df = pd.get_dummies(df.drop(columns="area"), columns=["month", "day"], dtype=float)
feature_names = list(X_df.columns)
X = X_df.values.astype(float)
print("Số mẫu, số biến:", X.shape)

np.random.seed(42)
idx = np.random.permutation(len(X))      
n_train = int(0.8 * len(X))

X_train, X_test = X[idx[:n_train]], X[idx[n_train:]]
y_train, y_test = y[idx[:n_train]], y[idx[n_train:]]

Số mẫu, số biến: (517, 29)


### Assignment 1

In [2]:
def add_bias(X):
    return np.hstack([np.ones((len(X), 1)), X])

class NewLinearRegression:
    def fit(self, X, y):
        Xb = add_bias(X)

        self.w = np.linalg.pinv(Xb.T @ Xb) @ Xb.T @ y

    def predict(self, X):
        return add_bias(X) @ self.w

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)

def evaluate(model, Xtr, Xte, name):
    print(f"{name:25s} | train MSE = {mse(y_train, model.predict(Xtr)):.4f}"
          f" | test MSE = {mse(y_test, model.predict(Xte)):.4f}"
          f" | test R2 = {r2(y_test, model.predict(Xte)):.4f}")

model1 = NewLinearRegression()
model1.fit(X_train, y_train)
evaluate(model1, X_train, X_test, "Dữ liệu gốc")

Xb = add_bias(X_train)
print("Số cột:", Xb.shape[1], "| Hạng của X:", np.linalg.matrix_rank(Xb))

Dữ liệu gốc               | train MSE = 1.8866 | test MSE = 1.6340 | test R2 = -0.1214
Số cột: 30 | Hạng của X: 27


### Assignment 2

In [3]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1          

X_train_s = (X_train - mean) / std
X_test_s = (X_test - mean) / std

model2 = NewLinearRegression()
model2.fit(X_train_s, y_train)

evaluate(model1, X_train, X_test, "Dữ liệu gốc")
evaluate(model2, X_train_s, X_test_s, "Dữ liệu đã chuẩn hóa")

Dữ liệu gốc               | train MSE = 1.8866 | test MSE = 1.6340 | test R2 = -0.1214
Dữ liệu đã chuẩn hóa      | train MSE = 1.8866 | test MSE = 1.6426 | test R2 = -0.1273


### Assignment 3

In [4]:
keep = []                                   
for j in range(X_train.shape[1]):
    cols = keep + [j]

    if np.linalg.matrix_rank(add_bias(X_train[:, cols])) == len(cols) + 1:
        keep.append(j)

removed = [feature_names[j] for j in range(X.shape[1]) if j not in keep]
print("Các cột bị loại:", removed)

X_train_p = X_train[:, keep]
X_test_p = X_test[:, keep]

model3 = NewLinearRegression()
model3.fit(X_train_p, y_train)

evaluate(model1, X_train, X_test, "Dữ liệu gốc")
evaluate(model3, X_train_p, X_test_p, "Đã loại cột cộng tuyến")

Các cột bị loại: ['month_nov', 'month_sep', 'day_wed']
Dữ liệu gốc               | train MSE = 1.8866 | test MSE = 1.6340 | test R2 = -0.1214
Đã loại cột cộng tuyến    | train MSE = 1.8866 | test MSE = 1.6718 | test R2 = -0.1473


### Assignment 4

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

sk_model = LinearRegression()
sk_model.fit(X_train_p, y_train)         
sk_pred = sk_model.predict(X_test_p)

print("scikit-learn : test MSE =", round(mean_squared_error(y_test, sk_pred), 4),
      "| test R2 =", round(r2_score(y_test, sk_pred), 4))
print("Tự cài       : test MSE =", round(mse(y_test, model3.predict(X_test_p)), 4),
      "| test R2 =", round(r2(y_test, model3.predict(X_test_p)), 4))

print("Chênh lệch lớn nhất ở beta_0:", abs(model3.w[0] - sk_model.intercept_))
print("Chênh lệch lớn nhất ở beta  :", np.max(np.abs(model3.w[1:] - sk_model.coef_)))

scikit-learn : test MSE = 1.6718 | test R2 = -0.1473
Tự cài       : test MSE = 1.6718 | test R2 = -0.1473
Chênh lệch lớn nhất ở beta_0: 5.85965720389936e-10
Chênh lệch lớn nhất ở beta  : 2.658868680782689e-10
